In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.preprocessing import LabelEncoder

import os
from PIL import Image
from tqdm import tqdm

# ------------------------------- 
# CONFIGURATION
# -------------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_TOP = 8
EPOCHS_FINE = 5

# Paths (exactly matching your folder structure)
FACE_TRAIN_DIR = 'data/face_train/'
FACE_TEST_DIR  = 'data/face_test/'
FLOWER_TEST_DIR = 'data/flower_test/'

# Create plots and models folders if not exist
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ------------------------------- 
# 1. Helper Functions
# -------------------------------
def load_and_preprocess_image(img_path):
    img = Image.open(img_path).convert('RGB')
    img = img.resize(IMG_SIZE)
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

def extract_features(feature_model, image_paths, batch_size=32):
    features = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting features"):
        batch_paths = image_paths[i:i+batch_size]
        batch_imgs = np.vstack([load_and_preprocess_image(p) for p in batch_paths])
        batch_features = feature_model.predict(batch_imgs, verbose=0)
        features.append(batch_features)
    return np.vstack(features)

def get_image_paths_and_labels(data_dir):
    image_paths = []
    labels = []
    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)
    return image_paths, labels

# ------------------------------- 
# 2. Load Pre-trained Feature Extractor (ImageNet)
# -------------------------------
print("Loading pre-trained ResNet50...")
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
feature_extractor_pretrained = Model(inputs=base_model.input, outputs=x)
print("Pre-trained feature extractor ready (2048-dim)")

# ------------------------------- 
# 3. Build & Fine-tune Model
# -------------------------------
def build_and_finetune(train_dataset, val_dataset, num_classes, task_name):
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    
    x = GlobalAveragePooling2D(name='top_gap')(base.output)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base.input, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    
    print(f"\n=== Training top layers for {task_name} ===")
    model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS_TOP, verbose=1)
    
    # Fine-tuning
    base.trainable = True
    for layer in base.layers[:-35]:   # Unfreeze last 35 layers
        layer.trainable = False
    
    model.compile(optimizer=Adam(learning_rate=1e-5), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    
    print(f"\n=== Fine-tuning {task_name} ===")
    model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS_FINE, verbose=1)
    
    return model

# ------------------------------- 
# 4. Face Recognition Task
# -------------------------------
print("\n" + "="*60)
print("FACE RECOGNITION TASK")
print("="*60)

train_ds_face = tf.keras.utils.image_dataset_from_directory(
    FACE_TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, 
    label_mode='categorical', validation_split=0.2, subset='training', seed=42)

val_ds_face = tf.keras.utils.image_dataset_from_directory(
    FACE_TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, 
    label_mode='categorical', validation_split=0.2, subset='validation', seed=42)

face_class_names = train_ds_face.class_names
num_face_classes = len(face_class_names)
print(f"Found {num_face_classes} face classes: {face_class_names}")

face_model = build_and_finetune(train_ds_face, val_ds_face, num_face_classes, "Face Recognition")

# Create fine-tuned feature extractor
face_feature_extractor_finetuned = Model(inputs=face_model.input, 
                                         outputs=face_model.get_layer('top_gap').output)

# Load your own test faces
face_test_paths, face_test_labels = get_image_paths_and_labels(FACE_TEST_DIR)
print(f"Loaded {len(face_test_paths)} test face images")

# Extract features
features_face_pre  = extract_features(feature_extractor_pretrained, face_test_paths)
features_face_post = extract_features(face_feature_extractor_finetuned, face_test_paths)

le_face = LabelEncoder()
y_face = le_face.fit_transform(face_test_labels)

# ------------------------------- 
# 5. Flower Recognition Task (using Oxford Flowers102 via tfds)
# -------------------------------
print("\n" + "="*60)
print("FLOWER RECOGNITION TASK")
print("="*60)

(ds_train, ds_val), info = tfds.load('oxford_flowers102', 
                                    split=['train', 'validation'], 
                                    with_info=True, as_supervised=True)

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)
    return image, tf.one_hot(label, depth=info.features['label'].num_classes)

train_ds_flower = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds_flower   = ds_val.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

num_flower_classes = info.features['label'].num_classes

flower_model = build_and_finetune(train_ds_flower, val_ds_flower, num_flower_classes, "Flower Recognition")

flower_feature_extractor_finetuned = Model(inputs=flower_model.input, 
                                           outputs=flower_model.get_layer('top_gap').output)

# Load your own flower test images
flower_test_paths, flower_test_labels = get_image_paths_and_labels(FLOWER_TEST_DIR)
print(f"Loaded {len(flower_test_paths)} test flower images")

features_flower_pre  = extract_features(feature_extractor_pretrained, flower_test_paths)
features_flower_post = extract_features(flower_feature_extractor_finetuned, flower_test_paths)

le_flower = LabelEncoder()
y_flower = le_flower.fit_transform(flower_test_labels)

# ------------------------------- 
# 6. Dimensionality Reduction & Visualization
# -------------------------------
def plot_embedding(features, labels, title, filename, reducer_name='PCA'):
    if reducer_name == 'PCA':
        reducer = PCA(n_components=2, random_state=42)
    elif reducer_name == 'TSNE':
        reducer = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features)-1))
    elif reducer_name == 'UMAP':
        reducer = umap.UMAP(n_components=2, random_state=42)
    
    embedding = reducer.fit_transform(features)
    
    plt.figure(figsize=(12, 9))
    sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], 
                    hue=labels, palette='tab10', s=80, alpha=0.8, edgecolor='k')
    plt.title(title, fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Classes')
    plt.tight_layout()
    plt.savefig(f'plots/{filename}.png', dpi=300, bbox_inches='tight')
    plt.show()

reducers = ['PCA', 'TSNE', 'UMAP']

print("\nGenerating plots...")

# Face Task
for red in reducers:
    plot_embedding(features_face_pre,  y_face, 
                   f'Face Recognition - Before Fine-tuning ({red})', 
                   f'face_before_{red.lower()}')
    plot_embedding(features_face_post, y_face, 
                   f'Face Recognition - After Fine-tuning ({red})', 
                   f'face_after_{red.lower()}')

# Flower Task
for red in reducers:
    plot_embedding(features_flower_pre,  y_flower, 
                   f'Flower Recognition - Before Fine-tuning ({red})', 
                   f'flower_before_{red.lower()}')
    plot_embedding(features_flower_post, y_flower, 
                   f'Flower Recognition - After Fine-tuning ({red})', 
                   f'flower_after_{red.lower()}')

# ------------------------------- 
# 7. Save Models
# -------------------------------
face_model.save('models/face_finetuned_resnet50.h5')
flower_model.save('models/flower_finetuned_resnet50.h5')

print("\n" + "="*60)
print("ASSIGNMENT-2 COMPLETED SUCCESSFULLY!")
print("All plots saved in 'plots/' folder")
print("Models saved in 'models/' folder")
print("="*60)

Loading pre-trained ResNet50...
Pre-trained feature extractor ready (2048-dim)

FACE RECOGNITION TASK
Found 25 files belonging to 4 classes.
Using 20 files for training.
Found 25 files belonging to 4 classes.
Using 5 files for validation.
Found 4 face classes: ['person1', 'person2', 'person3', 'person4']

=== Training top layers for Face Recognition ===
Epoch 1/8


I0000 00:00:1776269666.584313    5764 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_98327__.187


1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step - accuracy: 0.1500 - loss: 2.0696 - val_accuracy: 0.6000 - val_loss: 3.4800
Epoch 2/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step - accuracy: 0.6000 - loss: 1.3795 - val_accuracy: 0.6000 - val_loss: 3.9260
Epoch 3/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.6500 - loss: 1.1171 - val_accuracy: 0.6000 - val_loss: 4.2296
Epoch 4/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step - accuracy: 0.7500 - loss: 0.8256 - val_accuracy: 0.6000 - val_loss: 4.3148
Epoch 5/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.8500 - loss: 0.4332 - val_accuracy: 0.6000 - val_loss: 4.1376
Epoch 6/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step - accuracy: 0.8500 - loss: 0.4416 - val_accuracy: 0.6000 - val_loss: 3.3405
Epoch 7/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step - accuracy: 0.9500 - loss: 0.3925 - val_accuracy: 0.6000 - val_loss: 2.3906
Epoch 8/8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.9000 - loss: 0.2228 - val_accuracy: 0.6000 - val_loss: 1.3825

=== Fine-t

I0000 00:00:1776269709.252058    5764 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_116971__.231


1/1 ━━━━━━━━━━━━━━━━━━━━ 47s 47s/step - accuracy: 0.9500 - loss: 0.2990 - val_accuracy: 0.6000 - val_loss: 1.3710
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step - accuracy: 0.9500 - loss: 0.3119 - val_accuracy: 0.6000 - val_loss: 1.3580
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.8500 - loss: 0.3008 - val_accuracy: 0.6000 - val_loss: 1.3457
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.9500 - loss: 0.2000 - val_accuracy: 0.6000 - val_loss: 1.3347
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.9500 - loss: 0.1792 - val_accuracy: 0.6000 - val_loss: 1.3229
Loaded 33 test face images


...

Extracting features: 100%|██████████| 2/2 [00:09<00:00,  4.85s/it]



FLOWER RECOGNITION TASK


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

In [7]:
!pip install importlib_resources

In [3]:
!pip install tensorflow-datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 2.4 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 2.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 2.2 MB/s  0:00:21 eta 0:00:03
  Created wheel for promise: filename=promise-2.3-py3-none-any.whl size=21581 sha256=32bb5a7d6245f2e6956d48e9a984d96291d971706375b6ae97d63b012e324e6a
  Stored in directory: /home/nazmulhasan77/.cache/pip/wheels/8f/46/1c/1f4e5d73a20eb816ead5014e97cdeb3928cf314fc46c7bab61
Successfully built promise
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [tensorflow-datasets]nsorflow-datasets]
